# Agentic RAG

# NEXT LEVEL

It's time to take our Agentic RAG further, using a Tool to supplement our MCP servers.

I'm working on this Expert in questions about my courses - but you should apply this to your domain!

I've put a bunch of questions in `knowledge/faq.jsonl`

Please see the README for setup instructions.

In [1]:
import notebook_setup  # Windows Jupyter MCP fix — must run before agents import

from agents import Agent, Runner, SQLiteSession, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
import json
import sys
from IPython.display import display, Markdown
from pathlib import Path
from qdrant_client import QdrantClient
import gradio as gr

load_dotenv(override=True)

True

In [2]:
# Pick your model - start with gpt-5.4-mini

model = "gpt-5.5"

## The MCP Parameters

In [3]:
knowledge_dir = Path.cwd() / "knowledge"
knowledge_dir.mkdir(exist_ok=True)
vectordb_path = knowledge_dir / "vectordb"
faq_path = knowledge_dir / "faq.jsonl"

mcp_bin = Path(sys.executable).parent
qdrant_exe = mcp_bin / ("mcp-server-qdrant.exe" if sys.platform == "win32" else "mcp-server-qdrant")

vectorstore_params = {
    "command": str(qdrant_exe),
    "args": [],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

In [ ]:
client = QdrantClient(path=str(vectordb_path)) #! qdraant server 로 저장한 데이터 보기
collection_name = "knowledge"

info = client.get_collection(collection_name)
print(f"Memories in '{collection_name}': {info.points_count}\n")

points, _ = client.scroll(collection_name=collection_name, limit=200, with_payload=True, with_vectors=False)
for i, p in enumerate(points, 1):
    doc = (p.payload or {}).get("document", "")
    preview = doc.replace("\n", " ")[:160]
    print(f"{i:>3}. {preview}{'...' if len(doc) > 160 else ''}")

client.close()

Memories in 'knowledge': 57

  1. Untapt was selected for the Accenture FinTech Innovation Lab accelerator program and was recognized as an American Banker Top 20 Company To Watch.
  2. Ed Donner posted resources on May 28, 2025 titled 'Which order to take the AI courses?'.
  3. Nebula.io uses a patented AI model that matches people with roles without relying on keywords, offering greater accuracy and speed.
  4. Ed Donner created Udemy AI courses that have enrolled approximately 900,000 students.
  5. Edward Donner founded the AI startup untapt in 2013, building talent marketplaces and data science software for recruitment firms, initially focusing on tech ro...
  6. Ed Donner served as a Managing Director at JPMorgan before his AI startup ventures.
  7. Nebula.io’s mission is driven by the fact that 77% of people do not feel inspired or engaged at work, aiming to align individuals with their Ikigai.
  8. Ed Donner’s long‑term goal is to help people discover their purpose (Ikigai) and

In [5]:
with faq_path.open() as f:
    faqs = [json.loads(line) for line in f]

In [6]:
instructions = """
# Role

You are an expert about Ed Donner and his online courses. You are answering questions about him and his courses to visitors on his website.
Use your memories and tools to help find background information to answer the question. As needed, use multiple tools at the same time to gather all relevant context.
If you don't know the answer, say so.

# Memory

Always use your qdrant-find memory tool to help find relevant information. You can make multiple queries. Make all tool calls in parallel.

# FAQ

Your faq tool contains answers to all the common questions. Below is a list of the questions with their numbers.
If the user's question is related to one of these questions, then use your faq tool to retrieve a specific answer.
Respond with the answer in its original form in markdown, as written by Ed. If the answer include hyperlinks, then keep them in markdown format.

List of questions by number:
"""

for faq in faqs:
    instructions += f"\n{faq['faq']}. {faq['question']}"

In [7]:
display(Markdown(instructions))


# Role

You are an expert about Ed Donner and his online courses. You are answering questions about him and his courses to visitors on his website.
Use your memories and tools to help find background information to answer the question. As needed, use multiple tools at the same time to gather all relevant context.
If you don't know the answer, say so.

# Memory

Always use your qdrant-find memory tool to help find relevant information. You can make multiple queries. Make all tool calls in parallel.

# FAQ

Your faq tool contains answers to all the common questions. Below is a list of the questions with their numbers.
If the user's question is related to one of these questions, then use your faq tool to retrieve a specific answer.
Respond with the answer in its original form in markdown, as written by Ed. If the answer include hyperlinks, then keep them in markdown format.

List of questions by number:

1. In what order/sequence should I take Ed Donner's AI courses? Is there a recommended curriculum?
2. Can I take the courses with no Python or programming experience? Are the AI Engineering courses suitable for complete beginners, and what should I do if I'm new to coding?
3. Will completing your course qualify me to get a job as an AI Engineer? What else do I need to do?
4. I'm getting a Python NameError. What causes it and how do I fix it?
5. I'm getting a ModuleNotFoundError or ImportError (e.g. can't import openai or dotenv). How do I fix it?
6. My AI API key isn't being loaded from my .env file, isn't working, or load_dotenv(override=True) returns False. How do I fix it?
7. Why does Cursor show a stop sign with an 'AI features disabled' warning next to my .env file? Is something wrong?
8. Do I have to pay for APIs on this course, or can I use a free model, Gemini, OpenRouter, DeepSeek, Ollama, or another provider instead of OpenAI/Anthropic?
9. I'm getting an Archive Error while setting up my environment. How do I fix it?
10. I'm getting an error trying to run CrewAI. How do I fix it?
11. I'm having problems with uv (the package manager). How do I troubleshoot uv installation and sync issues?
12. (Agentic course) I'm having problems deploying my app to HuggingFace Spaces. How do I fix it?
13. Should I be concerned about privacy and data security when sending my data to frontier LLM providers like OpenAI and Anthropic?
14. I'm getting a 'Permission denied' error on a Mac when trying to install. How do I fix it?
15. I'm getting an SSL error, certificate error, network error, failure to download uv files, or an SSL-related API connection failure. How do I fix it?
16. Cursor isn't giving me AI autocomplete suggestions. How do I fix it?
17. I've finished my Cursor free trial and don't want to pay for it. What are my options?
18. The OpenAI API (or another AI provider) isn't working. How do I troubleshoot it, including quota and billing issues?
19. MCP isn't working in the Agentic AI course (e.g. Connection Refused or a timeout error). How do I fix it?
20. I'm getting a 'permission denied' error when trying to git push to the course repo. Why, and what should I do instead?
21. What do 'opinionated' and 'batteries included' mean when describing frameworks and libraries (e.g. CrewAI)?
22. I'm using VS Code (or another IDE) instead of Cursor and having trouble setting the kernel/environment, getting import errors. How do I fix it?
23. I'm getting an Internal Server Error (or other error) when deploying my application to Vercel in the production course. How do I debug it?
24. In Cursor or VS Code, I'm not seeing the same folders and files as in the videos. How do I open the project correctly?
25. I'm not able to select the Kernel for a Jupyter notebook in Cursor. How do I do it, including if the .venv environment doesn't appear?
26. I'm having problems with the MCP server mcp-memory-libsql. What's the alternative?
27. My Python script won't output Markdown (e.g. error: 'NoneType' object has no attribute 'display_id'). How can I display formatted output from a command-line script?
28. I'm having problems with n8n (in the Agentic Track course, not the AI Builder with n8n course). What should I do?
29. I'm having problems sending email with SendGrid. What alternatives can I use?
30. I'm getting a CrewAI error: AttributeError: module 'signal' has no attribute 'SIGHUP'. How do I fix it?
31. Will the Agentic AI course cover 'Skills' in addition to MCP?
32. Shouldn't we use the OpenAI Responses API instead of the Chat Completions API, since OpenAI recommends it?
33. I'm getting inconsistent performance from an LLM (tools not called reliably, unreliable agent flows, poor RAG performance, or inaccurate results). How do I make it more reliable?
34. Can we have a meeting (e.g. to pitch an idea, debug an issue, speak to your team, or get coaching)?
35. Can you fix my problem that involves 100 or more lines of code?
36. But can't I just use ChatGPT for that instead of building a specialized AI application?
37. Your courses come with projects and packages already set up. How do I create my own project from scratch, and how do I decide which packages to install?
38. (Core Track) I can't get Claude Code to work (typing 'claude' at the command line). What should I do?
39. How do I take a screenshot on Windows and on Mac, and what makes a good screenshot for troubleshooting?
40. Which AI architecture and frameworks should I use?
41. Which RAG technique should I use? (For example, in the Core Track, why can't I get an answer to 'Who went to Manchester University?')
42. I'm getting Rate Limit or Quota errors calling models via AWS Bedrock. How do I fix it?
43. (Core Track) What's the difference between using LLMs like OpenAI in weeks 1-2 and using LLMs with HuggingFace pipelines/transformers in week 3?
44. I'm having problems with Semgrep. What should I do?
45. I'm using free models on OpenRouter and getting RateLimit errors. How do I handle this?
46. With coding agents, what's the difference between CLI tools (like Claude Code) and visual IDE editors (like Cursor, or VS Code with the Claude Code extension), and how does this relate to which LLM is used?
47. (Agentic course Week 2) Why is the flow unreliable? Why does the Sales Manager send emails to the Sales Agents, and who selects the emails?
48. With the Equity Portfolio Rebalancer project, why am I getting inconsistent or unreliable results, such as prices not updating?
49. I'm building an agentic system. How do I prevent hallucinations and other alignment issues?
50. Cursor hangs when opening, and notebooks appear empty. How do I fix it?
51. AWS is no longer offering AWS App Runner to new customers. What should I do?
52. I don't want to spend money on coding agents, or I'm running out of usage quota. What are my options?
53. I've used up my quota with Cursor. How can I access free models?
54. In Cursor, my menus, screens, or options look different from yours. How do I get back to the normal view?
55. I'm getting an n8n error authenticating my OpenAI credentials: 'config.headers.setContentType is not a function'. How do I fix it?
56. Why haven't you replied to my LinkedIn connection request, message, or my tagging you?
57. In Cursor, my Kernel keeps crashing with an error mentioning 'OPENSSL_Applink', typically when connecting to OpenAI. How do I fix it?
58. I'm having problems with the video or audio in Udemy, perhaps on the mobile app. What should I do?

In [ ]:
faqs_lookup = {faq["faq"]: faq for faq in faqs}

def find_faq(question_number: int):
    faq = faqs_lookup.get(question_number)
    if faq:
        return f"### Question {faq['faq']} is:\n{faq['question']}\n### Ed's answer:\n{faq['answer']}"
    else:
        return "That question number was not found in the FAQ."   

@function_tool #! function 을 tool 로 바꾸기 - open ai sdk
def faq_tool(question_number:int ) -> str:
    """Use this tool to retrieve the answer to a frequently asked question by its number."""
    return find_faq(question_number)  

In [13]:
faq_tool.description

'Use this tool to retrieve the answer to a frequently asked question by its number.'

In [14]:
EXAMPLES = ["Which course covers RAG?", "Which course covers MCP?", "What job can I get after your courses?"]

In [15]:
convo = SQLiteSession("test_conversation")

## New and improved Agentic RAG!

In [16]:
# 사용자가 그냥 몇번째 질문을 받고 싶을때, 굳이 LLM 을 사용할 필요가 없음.
def has_instant_answer(message: str):
    stripped = message.strip().lower()
    return stripped.startswith("q") and stripped[1:].isdigit() and len(stripped) <= 3

def get_instant_answer(message: str):
    question_number = int(message.strip()[1:])
    return find_faq(question_number)

In [17]:
get_instant_answer("Q2")

"### Question 2 is:\nCan I take the courses with no Python or programming experience? Are the AI Engineering courses suitable for complete beginners, and what should I do if I'm new to coding?\n### Ed's answer:\nI've tried to design my courses to appeal to the widest possible audience - there should be something here for everyone.\n\nBut for the AI Engineering courses, I'm gonna be straight up: the courses are easiest for people with at least python foundations. It's entirely possible to take them with no python at all, and several people have completed everything and reported success and satisfaction. But it's definitely a steep learning curve.\n\nIf you're new to programming and decide to take the plunge with the AI Engineering courses, then:\n\n1. Please consider starting with the AI Builder course (first on the list of the [curriculum](https://edwarddonner.com/curriculum/)) as this is the best starting point for those new to coding\n2. Please spend time on the [self study guides](h

In [22]:
def text_from_event(event):
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        return event.data.delta
    elif event.type == "run_item_stream_event":
        if event.name == "tool_called":
            tool_name = getattr(event.item, "tool_name", None)
            return f'<small class="tool-status">Calling {tool_name}...</small>'
        elif event.name == "tool_output":
            return '<small class="tool-status">Tool returned. Thinking...</small>'

In [24]:
async def run(message):
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
        cumulative = ""
        agent = Agent(name="Expert", model=model, instructions=instructions, tools=[faq_tool], mcp_servers=[vectorstore_mcp])
        response = Runner.run_streamed(agent, message, session=convo)
        async for event in response.stream_events():
            if (text := text_from_event(event)):
                cumulative += text
                yield cumulative

### Minor change

I made one small addition since the video - can you see the line I added, and do you see why I needed to do that?

In [20]:
async def chat(message, history):
    if has_instant_answer(message):
        answer = get_instant_answer(message)
        await convo.add_items([{"role": "user", "content": message}, {"role": "assistant", "content": answer}])
        yield answer
    else:
        async for result in run(message):
            yield result

In [25]:
from styles import CSS, JS
gr.ChatInterface(chat, examples=EXAMPLES, chatbot=gr.Chatbot(show_label=False, height=800)).launch(css=CSS, js=JS, theme=gr.themes.Base(), inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\dobid\projects\expert\.venv\Lib\site-packages\mcp\os\win32\utilities.py", line 169, in create_windows_process
    process = await anyio.open_process(
              ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "c:\Users\dobid\projects\expert\.venv\Lib\site-packages\anyio\_core\_subprocesses.py", line 184, in open_process
    return await get_async_backend().open_process(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\dobid\projects\expert\.venv\Lib\site-packages\anyio\_backends\_asyncio.py", line 2617, in open_process
    process = await asyncio.create_subprocess_exec(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\dobid\AppData\Roaming\uv\python\cpython-3.13.5-windows-x86_64-none\Lib\asyncio\subprocess.py", line 224, in create_subprocess_exec
    transport, protocol = await loop.subprocess_exec(
             